#### Command Job submits

In [4]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

In [6]:
ws_config={
    "subscription_id": "7b1b43ca-4b64-43cf-9446-edb35a04d7d1",
    "resource_group": "AMLWS06",
    "workspace_name": "azuremlwest"
}

In [7]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

Found the config file in: .\config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [4]:
ml_client=MLClient(DefaultAzureCredential(
    additionally_allowed_tenants=["*"]),
                   ws_config['subscription_id'],
                   ws_config['resource_group'],
                   ws_config['workspace_name'],
                   )

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [6]:
from azure.ai.ml.entities import AzureBlobDatastore,AccountKeyConfiguration
account_key="Essf3gHrkhZgu6o1TuP74dizMbvtg3q7Xe1C5HPFs/UOddS0MF9oBSAi4WxckXMlRI81wLYjZ+ZL+AStU/T+RQ=="
blob_datastore=AzureBlobDatastore(
name="blob_diabetetes",
description="datastore-connection",
account_name='amlwstorage',
container_name='data',
credentials=AccountKeyConfiguration(
    account_key=account_key
)
)

In [8]:
#create datastore connection 
ml_client.create_or_update(blob_datastore)

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'blob_diabetetes', 'description': 'datastore-connection', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourceGroups/AMLWS06/providers/Microsoft.MachineLearningServices/workspaces/azuremlwest/datastores/blob_diabetetes', 'Resource__source_path': '', 'base_path': 'e:\\Azure DP100', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x000002EC343B7690>, 'credentials': {'type': 'account_key'}, 'container_name': 'data', 'account_name': 'amlwstorage', 'endpoint': 'core.windows.net', 'protocol': 'https'})

In [25]:
#path="https://amlwstorage.blob.core.windows.net/data"
path='azureml://datastores/blob_diabetetes/paths/'
#create data asset
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_asset=Data(
path=path,
type=AssetTypes.URI_FOLDER,
description="data_assetdiabetes",
name="diabetes_mltable"

)

In [26]:
ml_client.data.create_or_update(data_asset)

Data({'path': 'azureml://subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourcegroups/AMLWS06/workspaces/azuremlwest/datastores/blob_diabetetes/paths/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes_mltable', 'description': 'data_assetdiabetes', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourceGroups/AMLWS06/providers/Microsoft.MachineLearningServices/workspaces/azuremlwest/data/diabetes_mltable/versions/4', 'Resource__source_path': '', 'base_path': 'e:\\Azure DP100', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000002EC3454FA80>, 'serialize': <msrest.serialization.Serializer object at 0x000002EC364AD710>, 'version': '4', 'latest_version': None, 'datastore': None})

In [96]:
data_asset=ml_client.data.get('diabetes_mltable', version='4')

In [101]:
%%writefile train.py
import argparse
import pandas as pd 
import numpy as np 
#import mltable
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
#import mlflow
#from azure.ai.ml import command 
#from azure.ai.ml import MLClient
#from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from sklearn.preprocessing import LabelEncoder
import pickle
#import mlflow
import os 
parser=argparse.ArgumentParser()
parser.add_argument('--regularization', type=float, dest='reg_rate', default=0.01)
parser.add_argument('--training_data', type=str, dest='train')
args=parser.parse_args()

path=os.path.join(args.train, 'diabetes.csv')
df=pd.read_csv(path)
df.dropna(inplace=True)

y = df[['Outcome']]

x = df[['Pregnancies', 'Glucose', 'BloodPressure',
        'SkinThickness', 'Insulin', 'BMI',
        'DiabetesPedigreeFunction', 'Age']]
LE=LabelEncoder()
y=LE.fit_transform(y)
x_train, x_test, y_train, y_test=train_test_split(x,y, test_size=0.3)

model=LogisticRegression(C=1/args.reg_rate)
model.fit(x_train,y_train)
os.makedirs("outputs",exist_ok=True)
with open("outputs/model.pkl", "wb") as f:
        pickle.dump(model, f)
    #mlflow.sklearn.log_model(model, "model")
#mlflow.log_artifact("model.pkl")
y_hat=model.predict(x_test)

from sklearn.metrics import accuracy_score
acc=accuracy_score(y_hat, y_test)
print(f"Accuracy is {acc}")
#mlflow.log_metric("accuracy", acc)






Overwriting train.py


In [14]:
%%writefile train_env.yaml
name: train-env
channels:
- conda-forge
dependencies:
- python=3.10
- pip
- pip:
    - setuptools
    - mlflow==2.7.1
    - scikit-learn
    - pandas 
    - numpy 
    - azureml-core
    - azureml-dataset-runtime[fuse]
    - setuptools
    - azureml-fsspec

Overwriting train_env.yaml


In [60]:
%%writefile train_env.yaml

name: train-env
channels:
  - conda-forge
dependencies:
  - python=3.8.5
  - pip<22.0
  - pip:
    - pandas
    - azureml-core
    - azureml-dataset-runtime[fuse]
    - setuptools
    - scikit-learn
    - numpy
    - azureml-fsspec
    - mlflow
    - azureml-mlflow


Overwriting train_env.yaml


In [43]:
%%writefile infer_env.yaml

name: infer-env
channels:
  - conda-forge
dependencies:
  - python=3.8.5
  - pip<22.0
  - pip:
    - pandas
    - azureml-core
    - azureml-dataset-runtime[fuse]
    - setuptools
    - scikit-learn
    - numpy
    - azureml-fsspec
    - azureml-defaults
    - azureml-inference-server-http


Overwriting infer_env.yaml


In [61]:
from azure.ai.ml.entities import Environment

env=Environment(
name="custom-env-scikit-train",
image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
conda_file="train_env.yaml"

)

ml_client.environments.create_or_update(env)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'custom-env-scikit-train', 'description': None, 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourceGroups/AMLWS06/providers/Microsoft.MachineLearningServices/workspaces/azuremlwest/environments/custom-env-scikit-train/versions/1', 'Resource__source_path': '', 'base_path': 'e:\\Azure DP100', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000001B0886081F0>, 'serialize': <msrest.serialization.Serializer object at 0x000001B0885FDEF0>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.8.5', 'pip<22.0', {'pip': ['pandas', 'azureml-core', 'azureml-dataset-runtime[fuse]', 'setup

In [102]:
data_asset=ml_client.data.get('diabetes_mltable', version='4')
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.constants import AssetTypes, InputOutputModes

job=command(
code="./",
command="python train.py --regularization 0.03 --training_data ${{inputs.data}}",
 inputs={
            "data": Input(path=data_asset.id,
                type=AssetTypes.URI_FOLDER,
                mode=InputOutputModes.RO_MOUNT
            )
        },
        environment='custom-env-scikit:2',
        #"azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
        compute='satyakebakshi951'
    
)

In [74]:
!pip install setuptools

In [103]:
ml_client.jobs.create_or_update(job)

Uploading Azure DP100 (0.04 MBs): 100%|##########| 36594/36594 [00:00<00:00, 50913.83it/s]




Experiment,Name,Type,Status,Details Page
AzureDP100,calm_orange_mfdhxn35fz,command,Starting,Link to Azure Machine Learning studio


In [104]:
import os
os.system("docker --version")

0

### Batch Deployment

In [111]:
#deploy to batch endpoint

from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
model_name="diabetes-model-registered"
file_model=Model(
name=model_name,
path="model.pkl",
type=AssetTypes.CUSTOM_MODEL)

model=ml_client.models.create_or_update(file_model)

Uploading model.pkl (< 1 MB): 0.00B [00:00, ?B/s] (< 1 MB): 100%|##########| 970/970 [00:00<00:00, 11.1kB/s]




In [3]:
#deploy to batch endpoint
model=ml_client.models.get("diabetes-model-registered", version=1)
from azure.ai.ml.entities import BatchEndpoint
endpoint=BatchEndpoint(
    name="diabetes-batch-ep",
    description="batchendpoint"
)

ml_client.batch_endpoints.begin_create_or_update(endpoint)


In [6]:
endpoint.name

'diabetes-batch-ep'

In [118]:
%%writefile score.py
import json 
import numpy as np 
import pickle 
import os 
import pandas as pd
def init():
 global model
 
 model_path=os.path.join(os.getenv("AZUREML_MODEL_DIR"),'model.pkl')
 with open(model_path,"rb") as f:
    model=pickle.load(f)

def run(mini_batch):
  results=[]
  for file_path in mini_batch:
    df=pd.read_csv(file_path)
    X = df[['Pregnancies', 'Glucose', 'BloodPressure',
                'SkinThickness', 'Insulin', 'BMI',
                'DiabetesPedigreeFunction', 'Age']]
    preds=model.predict(X)
    df['Prediction']=preds
    results.append(df)
  return pd.concat(results)

Writing score.py


In [ ]:
from azure.ai.ml.entities import BatchDeployment, BatchRetrySettings,CodeConfiguration
from azure.ai.ml.constants import BatchDeploymentOutputAction
deployment=BatchDeployment(
name="diabetes-model-deployed",
description="diabetes_classifier",
endpoint_name="diabetes-batch-ep",
model=model,
compute='satyakebakshi951',
instance_count=1,
mini_batch_size=2,
code_configuration=CodeConfiguration(
    code='./',
    scoring_script="score.py"
),
environment="custom-env-scikit:6",

output_action=BatchDeploymentOutputAction.APPEND_ROW,
output_file_name="predictions_diabetes_batch.csv",
retry_settings=BatchRetrySettings(max_retries=3, timeout=300),
logging_level="info"

)


ml_client.batch_deployments.begin_create_or_update(deployment)

c:\Users\satya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\azure\ai\ml\entities\_deployment\batch_deployment.py:137: UserWarning: This class is intended as a base class and it's direct usage is deprecated. Use one of the concrete implementations instead:
* ModelBatchDeployment - For model-based batch deployments
* PipelineComponentBatchDeployment - For pipeline component-based batch deployments
  warnings.warn(
Uploading Azure DP100 (0.05 MBs): 100%|##########| 45852/45852 [00:00<00:00, 63644.38it/s] 




In [141]:
#create data asset
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

account_key="Essf3gHrkhZgu6o1TuP74dizMbvtg3q7Xe1C5HPFs/UOddS0MF9oBSAi4WxckXMlRI81wLYjZ+ZL+AStU/T+RQ=="
blob_datastore=AzureBlobDatastore(
name="blob_diabetetes_pred",
description="datastore-connection",
account_name='amlwstorage',
container_name='data-batch-pred',
credentials=AccountKeyConfiguration(
    account_key=account_key
)
)
ml_client.create_or_update(blob_datastore)

path='azureml://datastores/blob_diabetetes_pred/paths/'


data_asset=Data(
path=path,
type=AssetTypes.URI_FOLDER,
description="data_assetdiabetes",
name="diabetes_unlabeled"

)
ml_client.data.create_or_update(data_asset)


Data({'path': 'azureml://subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourcegroups/AMLWS06/workspaces/azuremlwest/datastores/blob_diabetetes_pred/paths/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes_unlabeled', 'description': 'data_assetdiabetes', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourceGroups/AMLWS06/providers/Microsoft.MachineLearningServices/workspaces/azuremlwest/data/diabetes_unlabeled/versions/2', 'Resource__source_path': '', 'base_path': 'e:\\Azure DP100', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000002EC3C2776D0>, 'serialize': <msrest.serialization.Serializer object at 0x000002EC3C288D00>, 'version': '2', 'latest_version': None, 'datastore': None})

In [2]:
patient_data_unlabeled=ml_client.data.get(
    name='diabetes_unlabeled', version=1
)

In [4]:
patient_data_unlabeled.id

'/subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourceGroups/AMLWS06/providers/Microsoft.MachineLearningServices/workspaces/azuremlwest/data/diabetes_unlabeled/versions/1'

In [1]:
from azure.identity import InteractiveBrowserCredential
from azure.ai.ml import MLClient

credential = InteractiveBrowserCredential(
    tenant_id="a287f42c-46eb-424f-ab40-cd784a7b423c"
)

ml_client = MLClient(
    credential,
    subscription_id="7b1b43ca-4b64-43cf-9446-edb35a04d7d1",
    resource_group_name="AMLWS06",
    workspace_name="azuremlwest"
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [4]:
print(endpoint.name)

diabetes-batch-ep


In [5]:
print(ml_client.workspace_name)

azuremlwest


In [6]:
print(ml_client.batch_endpoints.get(endpoint.name))

auth_mode: aad_token
defaults: {}
description: batchendpoint
id: /subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourceGroups/AMLWS06/providers/Microsoft.MachineLearningServices/workspaces/azuremlwest/batchEndpoints/diabetes-batch-ep
location: westus
name: diabetes-batch-ep
properties:
  BatchEndpointCreationApiVersion: '2023-10-01'
  azureml.onlineendpointid: /subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourceGroups/AMLWS06/providers/Microsoft.MachineLearningServices/workspaces/azuremlwest/batchEndpoints/diabetes-batch-ep
provisioning_state: Succeeded
scoring_uri: https://diabetes-batch-ep.westus.inference.ml.azure.com/jobs
tags: {}



In [ ]:
#submit the job

from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes
input=Input(type=AssetTypes.URI_FOLDER, path=patient_data_unlabeled.id)
job=ml_client.batch_endpoints.invoke(endpoint_name=endpoint.name, inputs={"input_data":input})



In [ ]:
#submit using CLI because of SDK issue
az ml batch-endpoint invoke \
  --name diabetes-batch-ep \
  --deployment-name diabetes-model-deployed \
  --input azureml:diabetes_unlabeled:2 \
  --resource-group AMLWS06 \
  --workspace-name azuremlwest

#### Online Endpoint or RLT Endpoint

In [22]:
from azure.ai.ml.entities import ManagedOnlineEndpoint,ManagedOnlineDeployment, CodeConfiguration
online_endpoint_name="MLonlineEndpoint-diabetes"
endpoint=ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="Online endpoint for diabetes model",
    auth_mode="key"

)

ml_client.online_endpoints.begin_create_or_update(endpoint)


In [9]:
model=ml_client.models.get("diabetes-model-registered", version=1)


In [38]:
%%writefile score_online_ep.py
import json 
import numpy as np 
import pickle 
import os 
import pandas as pd
def init():
 global model
 
 model_path=os.path.join(os.getenv("AZUREML_MODEL_DIR"),'model.pkl')
 with open(model_path,"rb") as f:
    model=pickle.load(f)

def run(raw_data):
  #results=[]
  #for file_path in mini_batch:
  data=json.loads(raw_data)
  data=np.array(data)
  #df=pd.read_csv(file_path)
  #df=pd.DataFrame(data)
  #X = df[['Pregnancies', 'Glucose', 'BloodPressure',
  #              'SkinThickness', 'Insulin', 'BMI',
  #              'DiabetesPedigreeFunction', 'Age']]
  preds=model.predict(data)
  #df['Prediction']=preds
  #results.append(df)
  return preds.tolist()#pd.concat(results)

Overwriting score_online_ep.py


In [54]:
%%writefile src/score_online_ep.py
import json 
import numpy as np 
import pickle 
import os 
import pandas as pd
def init():
 global model
 
 model_path=os.path.join(os.getenv("AZUREML_MODEL_DIR"),'model.pkl')
 with open(model_path,"rb") as f:
    model=pickle.load(f)

def run(raw_data):
  #results=[]
  #for file_path in mini_batch:
  data=json.loads(raw_data)
  #data=np.array(data)
  #df=pd.read_csv(file_path)
  df=pd.DataFrame(data)
  X = df[['Pregnancies', 'Glucose', 'BloodPressure',
                'SkinThickness', 'Insulin', 'BMI',
                'DiabetesPedigreeFunction', 'Age']]
  preds=model.predict(X)
  #df['Prediction']=preds
  #results.append(df)
  return preds.tolist()#pd.concat(results)

Overwriting src/score_online_ep.py


In [23]:
ml_client.online_endpoints.get(online_endpoint_name)

ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://mlonlineendpoint-diabetes.westus.inference.ml.azure.com/score', 'openapi_uri': 'https://mlonlineendpoint-diabetes.westus.inference.ml.azure.com/swagger.json', 'name': 'mlonlineendpoint-diabetes', 'description': 'Online endpoint for diabetes model', 'tags': {}, 'properties': {'createdBy': 'Satyake Bakshi', 'createdAt': '2026-03-05T19:28:43.337485+0000', 'lastModifiedAt': '2026-03-05T19:28:43.337485+0000', 'azureml.onlineendpointid': '/subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/resourcegroups/amlws06/providers/microsoft.machinelearningservices/workspaces/azuremlwest/onlineendpoints/mlonlineendpoint-diabetes', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/7b1b43ca-4b64-43cf-9446-edb35a04d7d1/providers/Microsoft.MachineLearningServices/locations/westus/mfeOperationsStatus/oeidp:291f085e-3ca2-45ec-9c74-12fbb2fc4c46:ac359867-42bf-4f85-9bfb-a6e3

In [46]:
import os
print(os.listdir("./"))

['config.json', 'diabetes.csv', 'train.py', 'train_env.yaml', 'model.pkl', 'score.py', 'Azure DP100 Notebook_2026.ipynb', 'score_online_ep.py', 'sample.JSON', 'infer_env.yaml']


In [ ]:

deployment=ManagedOnlineDeployment(
    name="blue",
    endpoint_name=online_endpoint_name,
    model=model,
    instance_type="Standard_DS1_v2",
    instance_count=1,
    code_configuration=CodeConfiguration(
    code="src", scoring_script="score_online_ep.py"
    ),
    environment='custom-env-scikit-infer:3'
)

ml_client.online_deployments.begin_create_or_update(deployment)

Instance type Standard_DS1_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint MLonlineEndpoint-diabetes exists
Uploading src (0.0 MBs): 100%|##########| 702/702 [00:00<00:00, 6885.28it/s]




................................................

In [49]:
#A/AB testing?
endpoint.traffic={'blue':100}
ml_client.begin_create_or_update(endpoint)

In [48]:
%%writefile sample.json 
[
  [6,148,72,35,0,33.6,0.627,50]
]

Overwriting sample.json


In [56]:
%%writefile sample1.json 

[
  {
    "Pregnancies": 6,
    "Glucose": 148,
    "BloodPressure": 72,
    "SkinThickness": 35,
    "Insulin": 0,
    "BMI": 33.6,
    "DiabetesPedigreeFunction": 0.627,
    "Age": 50
  }
]

Overwriting sample1.json


In [59]:
ml_client.online_endpoints.invoke(
    endpoint_name=online_endpoint_name,
    deployment_name="blue",
    request_file="sample1.json",
)

'[1]'

In [57]:
!az ml online-endpoint invoke --name MLOnlineEndpoint-diabetes --request-file sample1.json --resource-group AMLWS06 --workspace-name azuremlwest

"[1]"


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
